# Fase 3 — execução completa (todos os tópicos e subtópicos)

Notebook curto que só roda o pipeline pros 40 subtópicos, reaproveitando tudo
que o piloto (`pipeline_mcq_fase3.ipynb`) já calibrou e já processou: o
`out_dir` é o mesmo (`saida_fase3/`), então o cache de facetas, a varredura do
corpus, os embeddings, os codebooks e o repositório (com as 249 questões do
piloto já recuperadas) continuam valendo — este notebook **estende**, não
recomeça.

Pré-requisito: já ter rodado o piloto pelo menos até a calibração do limiar de
entropia (§14 de `pipeline_mcq_fase3.ipynb`). Se `saida_fase3/estado/` não
tiver os subtópicos do piloto, a célula de calibração avisa e cai no default
do `Config` (0,010 — já bem próximo do que o piloto calibrou, 0,009-0,012).

**Custo do que este notebook faz de novo**, em relação ao piloto: facetas para
os subtópicos que faltam (chamadas leves, com cache — pula os que já têm
faceta extraída), **uma nova varredura do corpus** (a assinatura inclui o
conjunto de facetas, então passar de 3 pra 40 subtópicos sempre dispara uma
nova passada, ~8 min com os três arquivos), e a execução em si, agora
proporcional a 40 subtópicos em vez de 3 — é a célula de execução completa que
concentra a maior parte do custo em chamadas de LLM. Se quiser ver
comportamento/custo antes de soltar nos 40, rode uma vez com `SUBTOPICOS`
reduzido a um punhado de subtópicos novos.

## 1. Setup

In [ ]:
import sys, json, glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

AQUI = Path.cwd()                      # pipeline/fase_3
RAIZ = (AQUI / ".." / "..").resolve()  # raiz do repositório
sys.path.insert(0, str(AQUI))
sys.path.insert(0, str(RAIZ))

import utils_fase3 as U
import prompts_fase3 as P
import seed_fase3 as S
from topicos import TOPICOS
from azure_openai_backend import AzureOpenAIBackend

pd.set_option("display.max_colwidth", 90)
print(f"raiz: {RAIZ}")
print(f"tópicos: {len(TOPICOS)} · subtópicos: {sum(len(v) for v in TOPICOS.values())}")

In [ ]:
cfg = U.Config(
    raiz=RAIZ,
    corpus_dir=RAIZ / "dataset" / "corpus-SemProcessamento-publico-PetrolesCompleto",
    out_dir=AQUI / "saida_fase3",   # MESMO out_dir do piloto — reaproveita tudo já calculado
    modelo_forte="gpt-5-4-petrobras",
    modelo_leve="gpt-5-mini-petrobras",
    # limiar_ganho_entropia fica no default do Config (0.010) até a célula de
    # calibração logo abaixo, que lê o histórico do piloto e ajusta se achar
    # dado suficiente.
)
print(cfg.resumo())

In [ ]:
# Um deployment por papel. O backend cacheia em disco por hash do pedido, então
# reexecutar o notebook não repaga as chamadas idênticas.
llm_leve = AzureOpenAIBackend(deployment=cfg.modelo_leve, max_tokens=3000)
llm_gerador = AzureOpenAIBackend(deployment=cfg.modelo_forte, max_tokens=12000,
                                 reasoning_effort=cfg.gerador_reasoning_effort)
llm_judge = AzureOpenAIBackend(deployment=cfg.modelo_forte, max_tokens=8000,
                               reasoning_effort=cfg.judge_reasoning_effort)
# O refinador é o mesmo gpt-5 do gerador — reescrever questão é tarefa de autor.
llm_refinador = llm_gerador

llm_leve.doctor()

## 2. Insumos: banco seed e escopo completo

In [ ]:
DIR_FASE2 = RAIZ / "pipeline" / "fase_2" / "questionarios" / "respostas_questionario"

banco_seed = S.construir_banco_seed(DIR_FASE2, min_nota_humana=0.5)
S.salvar_banco_seed(banco_seed, cfg.out_dir / "banco_seed.jsonl")
print(f"{len(banco_seed)} questões no banco seed")

In [ ]:
# Todos os tópicos/subtópicos — a única diferença de escopo em relação ao
# piloto (que rodava só 3) é esta linha.
SUBTOPICOS = [(t, s) for t, subs in TOPICOS.items() for s in subs]
print(f"{len(SUBTOPICOS)} subtópicos em {len(TOPICOS)} tópicos")

## 3. Limiar de entropia calibrado no piloto

Lê o histórico de entropia dos subtópicos que já rodaram (salvo em
`saida_fase3/estado/`) e sugere o limiar do critério de parada, do jeito que o
notebook principal faz na §14 (`U.sugerir_limiar_entropia`). Se não achar
nenhum estado ainda, mantém o default do `Config` (0,010) e avisa.

Ressalva do bug de perda de dados de ago/2026 (ver memória do projeto): o
histórico do piloto tem um ponto de transição — a rodada em que o repositório
foi recuperado — que é um outlier no ganho de entropia. A função usa percentil
75, então é razoavelmente robusta a esse único ponto, mas o número calibrado
(0,009-0,012 na última checagem) é uma referência, não uma verdade absoluta.

In [ ]:
historicos = {}
for p in glob.glob(str(cfg.out_dir / "estado" / "*.json")):
    est = json.loads(Path(p).read_text(encoding="utf-8"))
    if len(est.get("historico_entropia", [])) >= 3:
        historicos[est["subtopico"]] = est["historico_entropia"]

if historicos:
    limiar = U.sugerir_limiar_entropia(historicos, percentil=75)
    cfg.limiar_ganho_entropia = limiar
    print(f"\nadotando limiar calibrado: {cfg.limiar_ganho_entropia:.4f}")
else:
    print(f"nenhum estado de piloto encontrado em {cfg.out_dir / 'estado'} — "
          f"mantendo o default do Config: {cfg.limiar_ganho_entropia:.4f}")

## 4. Facetas, varredura do corpus, embeddings e planos — para TODOS os subtópicos

A varredura do corpus (célula seguinte) é a única etapa cara que roda de novo
inteira: a assinatura inclui o conjunto de facetas, e o conjunto mudou de 3
pra até 40 subtópicos.

In [ ]:
facetas = U.extrair_facetas(llm_leve, TOPICOS, cfg, subtopicos_alvo=SUBTOPICOS)
por_sub = U.agrupar_por_subtopico(facetas)
print(f"\n{len(facetas)} facetas em {len(por_sub)} subtópicos")

In [ ]:
%%time
U.descrever_corpus(cfg)
U.varrer_corpus(facetas, cfg)          # ~8 min com os três arquivos do corpus
candidatos = U.carregar_candidatos(cfg)
print(f"\n{sum(len(v) for v in candidatos.values()):,} candidatos em {len(candidatos)} facetas")

In [ ]:
emb = U.Embedder(cfg)

planos = {}
for subtopico, fs in por_sub.items():
    planos[subtopico] = U.plano_de_documentos(fs, candidatos, emb, cfg)
print(f"planos montados para {len(planos)} subtópicos · "
      f"{sum(len(p) for p in planos.values())} documentos no total")

## 5. Tolerâncias de vício e codebooks de entropia — para TODOS os subtópicos

In [ ]:
sugerido = U.calibrar_tolerancias(banco_seed, emb, cfg, percentil=75)
cfg.tol_similaridade = sugerido["similaridade"]
cfg.tol_comprimento  = sugerido["comprimento"]
cfg.tol_distratores  = sugerido["distratores"]

In [ ]:
codebooks = {}
for subtopico, fs in por_sub.items():
    codebooks[subtopico] = U.codebook_do_subtopico(subtopico, fs, candidatos, emb, cfg)
print(f"{len(codebooks)} codebooks prontos")

## 6. Repositório, pool de few-shot e execução completa

In [ ]:
repo = U.Repositorio(cfg, emb)
pool = U.PoolFewShot(banco_seed, repo, cfg)
print(f"repositório: {len(repo)} questões já armazenadas (piloto incluso)")
print("pool de few-shot:", pool.composicao())

`executar_subtopico` encadeia geração → vícios → refinamento → judge →
armazenamento → entropia (judge por último, ver memória do projeto), avança de
documento quando o subtópico satura e termina quando a fila de documentos
acaba. É retomável: os subtópicos do piloto já processados continuam de onde
pararam (a checagem de consistência do início de `executar_subtopico` confere
isso antes de seguir); os subtópicos novos começam do zero.

`max_rodadas` fica no default da própria função (60) — quem decide quando cada
subtópico termina é o critério de parada por entropia, não esse teto (ele só
evita um loop sem fim). `repo.salvar()` a cada subtópico concluído, não só no
final — com 40 subtópicos essa célula pode rodar por muito tempo, e as
questões em si já são gravadas incrementalmente (`Repositorio.adicionar`)
conforme aprovadas, mas os embeddings pesam menos gravados com mais frequência.

> Célula cara. Considere reduzir `SUBTOPICOS` a um punhado antes de soltar nos
> 40, se ainda não tiver visto o comportamento/custo desta versão do pipeline
> (judge no fim do loop) em produção.

In [ ]:
%%time
estados = {}
for topico, subtopico in SUBTOPICOS:
    estados[subtopico] = U.executar_subtopico(
        subtopico=subtopico, topico=topico,
        facetas=por_sub[subtopico], plano=planos[subtopico],
        llm_leve=llm_leve, llm_forte=llm_gerador, llm_judge=llm_judge,
        pool=pool, repo=repo, emb=emb, codebook=codebooks[subtopico],
        cfg=cfg,
    )
    repo.salvar()

print(f"\nrepositório: {len(repo)} questões · pool: {pool.composicao()}")

## 7. Export final

In [ ]:
destino = cfg.out_dir / "questoes_fase3.jsonl"
with open(destino, "w", encoding="utf-8") as fh:
    for q in repo.questoes:
        fh.write(json.dumps(q, ensure_ascii=False) + "\n")
print(f"{len(repo)} questões -> {destino}")
print(f"embeddings           -> {repo.caminho_emb}")
print(f"estado por subtópico -> {cfg.out_dir / 'estado'}")
print(f"log de rodadas       -> {cfg.out_dir / 'logs' / 'rodadas.jsonl'}")
print(f"\nuso de tokens:")
for nome, llm in [("leve", llm_leve), ("gerador", llm_gerador), ("judge", llm_judge)]:
    u = llm.usage
    print(f"  {nome:<8} {u.calls:>4} chamadas ({u.cached_calls} de cache) · "
          f"{u.prompt_tokens:>9,} in · {u.completion_tokens:>8,} out")

## 8. O que revisar antes de considerar o lote pronto

1. os documentos consolidados que começam com `AVISO:` — recuperação fraca
   naquela faceta, e questão gerada dali não presta;
2. a taxa de descarte do judge por subtópico — descarte alto costuma ser
   documento ruim, não gerador ruim;
3. a distribuição de dificuldade — se o judge estiver marcando quase tudo como
   "media", a escala não está discriminando;
4. rodar de novo `analysis/avaliacao_dificuldade_ollama.ipynb` sobre o lote
   completo, pra comparar a fração de questões "Fácil" com a 1ª rodada (87%).